# Optimisation CPU des Transformers sur LIAR

Ce notebook vise à maximiser le F1 pondéré de DistilBERT et RoBERTa sur le dataset LIAR, en environnement **CPU-only**.

Les étapes suivent la stratégie : tuning rapide sur sous-ensemble, puis run finale sur tout le train avec la meilleure config.


In [43]:
# Charger les variables d'environnement depuis un fichier .env (sécurisé)
from dotenv import load_dotenv
import os
from huggingface_hub import login

load_dotenv(dotenv_path=".env")  # Charge le .env du dossier courant

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("HF_TOKEN chargé")
else:
    print("Aucun token HF_TOKEN trouvé")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF_TOKEN chargé


In [44]:
# Variables globales pour tuning et run finale
USE_SUBSET = True  # True = tuning rapide, False = run finale
SUBSET_SIZE_TRAIN = 2000
SUBSET_SIZE_VAL = 500
SUBSET_SIZE_TEST = 500

# Pour la gestion du déséquilibre
USE_SAMPLER = False  # False = class_weight dans la loss, True = WeightedRandomSampler dans le DataLoader


## Chargement des données LIAR avec gestion du sous-ensemble

Toutes les fonctions de chargement doivent respecter les variables `USE_SUBSET`, `SUBSET_SIZE_TRAIN`, `SUBSET_SIZE_VAL`, `SUBSET_SIZE_TEST`.


In [45]:
import pandas as pd
import numpy as np
from pathlib import Path

# Chemins des fichiers parquet
DATA_DIR = Path("../data/traitees")
TRAIN_PATH = DATA_DIR / "liar_train.parquet"
VAL_PATH = DATA_DIR / "liar_val.parquet"
TEST_PATH = DATA_DIR / "liar_test.parquet"

# Chargement avec gestion du sous-ensemble
def load_liar_df(path, subset_size=None):
    df = pd.read_parquet(path)
    if subset_size is not None and len(df) > subset_size:
        df = df.sample(n=subset_size, random_state=42).reset_index(drop=True)
    # Correction : forcer label_binary en int
    if "label_binary" in df.columns:
        df["label_binary"] = df["label_binary"].astype(int)
    return df

train_df = load_liar_df(TRAIN_PATH, SUBSET_SIZE_TRAIN if USE_SUBSET else None)
val_df = load_liar_df(VAL_PATH, SUBSET_SIZE_VAL if USE_SUBSET else None)
test_df = load_liar_df(TEST_PATH, SUBSET_SIZE_TEST if USE_SUBSET else None)

print(f"train_df: {train_df.shape}, val_df: {val_df.shape}, test_df: {test_df.shape}")
print("Labels train:", train_df['label_binary'].value_counts(normalize=True))
print("Labels val:", val_df['label_binary'].value_counts(normalize=True))
print("Labels test:", test_df['label_binary'].value_counts(normalize=True))

train_df: (2000, 18), val_df: (500, 18), test_df: (500, 18)
Labels train: label_binary
1    0.566
0    0.434
Name: proportion, dtype: float64
Labels val: label_binary
1    0.512
0    0.488
Name: proportion, dtype: float64
Labels test: label_binary
1    0.574
0    0.426
Name: proportion, dtype: float64


In [ ]:
# Mapping binaire LIAR : vrai = 1, faux = 0
label_map = {
    "true": 1,
    "mostly-true": 1,
    "half-true": 1,
    "false": 0,
    "barely-true": 0,
    "pants-fire": 0
}
for df in [train_df, val_df, test_df]:
    if "label" in df.columns:
        df["label"] = df["label"].map(label_map)
        assert df["label"].isnull().sum() == 0, "Des labels n'ont pas été convertis !"
        df["label"] = df["label"].astype(int)
print("Mapping binaire appliqué. Exemples:", train_df["label"].value_counts())

In [46]:
# Préparation des entrées texte seul et texte+méta

def concat_meta(row):
    statement = str(row.get("statement", "UNKNOWN"))
    subject = str(row.get("subject", "UNKNOWN"))
    party = str(row.get("party", "UNKNOWN"))
    job = str(row.get("job_title", "UNKNOWN"))
    speaker = str(row.get("speaker", "UNKNOWN"))
    return (
        f"[CLAIM] {statement} "
        f"[META] subject={subject}; party={party}; job={job}; speaker={speaker}"
    )

for df in [train_df, val_df, test_df]:
    df["text_input_A"] = df["statement"].astype(str)
    df["text_input_B"] = df.apply(concat_meta, axis=1)

print("Exemple text_input_A:", train_df["text_input_A"].iloc[0])
print("Exemple text_input_B:", train_df["text_input_B"].iloc[0])

Exemple text_input_A: Polling shows that nearly 74 percent of National Rifle Association members support requiring background checks for all gun sales.
Exemple text_input_B: [CLAIM] Polling shows that nearly 74 percent of National Rifle Association members support requiring background checks for all gun sales. [META] subject=civil-rights,government-regulation,guns; party=democrat; job=State Senator, District 4; speaker=lena-taylor


## Implémentation de DistilBERT pour la classification de texte
Ce modèle utilise la version pré-entraînée `distilbert-base-uncased` de Hugging Face pour effectuer une classification binaire sur les textes du dataset LIAR.

In [47]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset

# Charger le tokenizer et le modèle DistilBERT pré-entraîné
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Préparer le Dataset HuggingFace à partir du DataFrame
# On suppose que train_df, val_df, test_df ont déjà les colonnes 'text_input_A' et 'label' (int)
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

def tokenize_function(examples):
    return tokenizer(examples["text_input_A"], truncation=True, padding="max_length", max_length=64)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("Exemple d'entrée tokenizée :", train_dataset[0])

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Exemple d'entrée tokenizée : {'id': '10626.json', 'label': 'true', 'statement': 'Polling shows that nearly 74 percent of National Rifle Association members support requiring background checks for all gun sales.', 'subject': 'civil-rights,government-regulation,guns', 'speaker': 'lena-taylor', 'job_title': 'State Senator, District 4', 'state_info': 'Wisconsin', 'party': 'democrat', 'barely_true_counts': 1.0, 'false_counts': 1.0, 'half_true_counts': 0.0, 'mostly_true_counts': 1.0, 'pants_on_fire_counts': 1.0, 'context': 'a news release', 'split': 'train', 'label_binary': 1, 'n_chars': 129, 'n_tokens': 19, 'text_input_A': 'Polling shows that nearly 74 percent of National Rifle Association members support requiring background checks for all gun sales.', 'text_input_B': '[CLAIM] Polling shows that nearly 74 percent of National Rifle Association members support requiring background checks for all gun sales. [META] subject=civil-rights,government-regulation,guns; party=democrat; job=State Sena

## Fonctions utilitaires pour la gestion du déséquilibre


In [48]:
from torch.utils.data import WeightedRandomSampler
import torch

def get_class_weights(labels):
    class_sample_count = np.array([np.sum(labels == t) for t in np.unique(labels)])
    weight = 1. / class_sample_count
    class_weights = {cls: w for cls, w in zip(np.unique(labels), weight)}
    return torch.tensor([class_weights[cls] for cls in labels], dtype=torch.float)

def get_weighted_sampler(labels):
    class_sample_count = np.array([np.sum(labels == t) for t in np.unique(labels)])
    weight = 1. / class_sample_count
    samples_weight = np.array([weight[t] for t in labels])
    samples_weight = torch.from_numpy(samples_weight).double()
    sampler = WeightedRandomSampler(samples_weight, len(samples_weight), replacement=True)
    return sampler

# Exemple d'utilisation
class_weights = get_class_weights(train_df['label_binary'].values)
sampler = get_weighted_sampler(train_df['label_binary'].values)
print("Class weights:", class_weights[:5])
print("Sampler example:", list(sampler)[:5])

Class weights: tensor([0.0009, 0.0012, 0.0012, 0.0009, 0.0012])
Sampler example: [1350, 485, 477, 383, 291]


## Entraînement DistilBERT (texte seul) : tuning hyperparamètres et gestion du déséquilibre


In [49]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import torch
import numpy as np

# Tokenizer et model_name
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenization
max_length = 64
def tokenize_function(examples):
    return tokenizer(examples["text_input_A"], truncation=True, padding="max_length", max_length=max_length)

# Correction : renommer label_binary en label pour HuggingFace Dataset
for df in [train_df, val_df, test_df]:
    # Supprimer les colonnes dupliquées
    df = df.loc[:, ~df.columns.duplicated()]
    if "label_binary" in df.columns:
        df.rename(columns={"label_binary": "label"}, inplace=True)
    # S'assurer qu'il n'y a qu'une seule colonne 'label'
    if df.columns.tolist().count("label") > 1:
        first = True
        cols = []
        for col in df.columns:
            if col == "label" and first:
                cols.append(col)
                first = False
            elif col != "label":
                cols.append(col)
        df = df[cols]
    # Correction finale : convertir 'true'/'false' en 1/0 puis forcer label en int
    if "label" in df.columns:
        df["label"] = df["label"].replace({"true": 1, "false": 0, "True": 1, "False": 0})
        df["label"] = df["label"].astype(int)

# Dataset HuggingFace
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df.loc[:, ~train_df.columns.duplicated()])
val_dataset = Dataset.from_pandas(val_df.loc[:, ~val_df.columns.duplicated()])
test_dataset = Dataset.from_pandas(test_df.loc[:, ~test_df.columns.duplicated()])
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Compute metrics
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    f1w = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"f1_weighted": f1w, "accuracy": acc}


ValueError: invalid literal for int() with base 10: 'barely-true'

In [50]:
# Afficher les valeurs uniques de la colonne label AVANT conversion
for df, name in zip([train_df, val_df, test_df], ["train", "val", "test"]):
    if "label" in df.columns:
        print(f"Valeurs uniques dans {name}_df['label']:", df["label"].unique())

Valeurs uniques dans train_df['label']: <ArrowStringArray>
['true', 'barely-true', 'false', 'mostly-true', 'half-true', 'pants-fire']
Length: 6, dtype: str
Valeurs uniques dans val_df['label']: <ArrowStringArray>
['false', 'mostly-true', 'true', 'barely-true', 'half-true', 'pants-fire']
Length: 6, dtype: str
Valeurs uniques dans test_df['label']: <ArrowStringArray>
['half-true', 'false', 'mostly-true', 'pants-fire', 'barely-true', 'true']
Length: 6, dtype: str


In [51]:
# Mapping binaire LIAR : vrai = 1, faux = 0
label_map = {
    "true": 1,
    "mostly-true": 1,
    "half-true": 1,
    "false": 0,
    "barely-true": 0,
    "pants-fire": 0
}
for df in [train_df, val_df, test_df]:
    if "label" in df.columns:
        df["label"] = df["label"].map(label_map)
        assert df["label"].isnull().sum() == 0, "Des labels n'ont pas été convertis !"
        df["label"] = df["label"].astype(int)
print("Mapping binaire appliqué. Exemples:", train_df["label"].value_counts())

Mapping binaire appliqué. Exemples: label
1    1132
0     868
Name: count, dtype: int64


In [ ]:
from transformers import TrainingArguments
print(TrainingArguments)
help(TrainingArguments)

<class 'transformers.training_args.TrainingArguments'>
Help on class TrainingArguments in module transformers.training_args:

class TrainingArguments(builtins.object)
 |  TrainingArguments(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = 

In [ ]:
# Boucle de tuning : 2 LR x 2 schémas de déséquilibre
lrs = [2e-5, 3e-5]
disbal_methods = ["class_weight", "sampler"]
results = []

for lr in lrs:
    for method in disbal_methods:
        print(f"\n==== Tuning: LR={lr}, méthode={method} ====")
        use_sampler = (method == "sampler")
        # Model
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
        # TrainingArguments
        args = TrainingArguments(
            output_dir="./results_distilbertA",
            num_train_epochs=6,
            per_device_train_batch_size=8,
            eval_strategy="epoch",  # <-- correction ici
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="f1_weighted",
            warmup_ratio=0.1,
            learning_rate=lr,
            report_to="none",  # <-- correction ici
            seed=42
        )
        # Gestion du déséquilibre
        if use_sampler:
            sampler = get_weighted_sampler(train_df['label_binary'].values)
            trainer = Trainer(
                model=model,
                args=args,
                train_dataset=train_dataset,
                eval_dataset=val_dataset,
                compute_metrics=compute_metrics,
                callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
                data_collator=None,
                sampler=sampler
            )
        else:
            class_weights = get_class_weights(train_df['label_binary'].values)
            trainer = Trainer(
                model=model,
                args=args,
                train_dataset=train_dataset,
                eval_dataset=val_dataset,
                compute_metrics=compute_metrics,
                callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
                data_collator=None,
                # On passe les class_weights dans la loss
                # (nécessite d'adapter la loss dans le modèle si besoin)
            )
        # Entraînement
        train_result = trainer.train()
        metrics = trainer.evaluate()
        print("Résultats val:", metrics)
        results.append({"lr": lr, "method": method, "f1w_val": metrics["eval_f1_weighted"], "acc_val": metrics["eval_accuracy"]})


==== Tuning: LR=2e-05, méthode=class_weight ====


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
c:\Users\conta\Documents\grp3_projet3_data\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as tr

ValueError: too many dimensions 'str'

In [ ]:
# Résumé des résultats tuning
import pandas as pd
results_df = pd.DataFrame(results)
display(results_df.sort_values("f1w_val", ascending=False))

## Run finale DistilBERT texte seul (meilleure config)


In [ ]:
# Repasse en mode full data
USE_SUBSET = False
train_df = load_liar_df(TRAIN_PATH, None)
val_df = load_liar_df(VAL_PATH, None)
test_df = load_liar_df(TEST_PATH, None)

for df in [train_df, val_df, test_df]:
    df["text_input_A"] = df["statement"].astype(str)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Utilise la meilleure config trouvée (à adapter selon results_df)
best_lr = results_df.sort_values("f1w_val", ascending=False).iloc[0]["lr"]
best_method = results_df.sort_values("f1w_val", ascending=False).iloc[0]["method"]
use_sampler = (best_method == "sampler")

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
args = TrainingArguments(
    output_dir="./results_distilbertA_final",
    num_train_epochs=6,
    per_device_train_batch_size=8,
    eval_strategy="epoch",  # <-- correction ici
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    warmup_ratio=0.1,
    learning_rate=best_lr,
    report_to="none",
    device="cpu",  # <-- correction ici
    seed=42
)
if use_sampler:
    sampler = get_weighted_sampler(train_df['label_binary'].values)
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        data_collator=None,
        sampler=sampler
    )
else:
    class_weights = get_class_weights(train_df['label_binary'].values)
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        data_collator=None,
        # class_weights à intégrer dans la loss si besoin
    )
train_result = trainer.train()
metrics = trainer.evaluate(test_dataset)
print("Résultats test:", metrics)

## RoBERTa texte seul : tuning et run finale


In [ ]:
# Tuning RoBERTa texte seul
model_name_roberta = "roberta-base"
tokenizer_roberta = AutoTokenizer.from_pretrained(model_name_roberta)

def tokenize_function_roberta(examples):
    return tokenizer_roberta(examples["text_input_A"], truncation=True, padding="max_length", max_length=max_length)

train_dataset_roberta = Dataset.from_pandas(train_df)
val_dataset_roberta = Dataset.from_pandas(val_df)
test_dataset_roberta = Dataset.from_pandas(test_df)
train_dataset_roberta = train_dataset_roberta.map(tokenize_function_roberta, batched=True)
val_dataset_roberta = val_dataset_roberta.map(tokenize_function_roberta, batched=True)
test_dataset_roberta = test_dataset_roberta.map(tokenize_function_roberta, batched=True)

# Hyperparams
lr_roberta = 1e-5
model_roberta = AutoModelForSequenceClassification.from_pretrained(model_name_roberta, num_labels=2)
args_roberta = TrainingArguments(
    output_dir="./results_robertaA",
    num_train_epochs=6,
    per_device_train_batch_size=8,
    eval_strategy="epoch",  # <-- correction ici
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    warmup_ratio=0.1,
    learning_rate=lr_roberta,
    report_to="none",
    device="cpu",  # <-- correction ici
    seed=42
)
# Utilise la meilleure méthode de déséquilibre trouvée pour DistilBERT
if use_sampler:
    sampler = get_weighted_sampler(train_df['label_binary'].values)
    trainer_roberta = Trainer(
        model=model_roberta,
        args=args_roberta,
        train_dataset=train_dataset_roberta,
        eval_dataset=val_dataset_roberta,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        data_collator=None,
        sampler=sampler
    )
else:
    class_weights = get_class_weights(train_df['label_binary'].values)
    trainer_roberta = Trainer(
        model=model_roberta,
        args=args_roberta,
        train_dataset=train_dataset_roberta,
        eval_dataset=val_dataset_roberta,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        data_collator=None,
        # class_weights à intégrer dans la loss si besoin
    )
train_result_roberta = trainer_roberta.train()
metrics_roberta = trainer_roberta.evaluate(test_dataset_roberta)
print("Résultats test RoBERTa:", metrics_roberta)

## Expériences texte + métadonnées (DistilBERT et RoBERTa)


In [ ]:
# Tokenization pour texte+méta
max_length_B = 80  # Pour éviter de tronquer le statement

def tokenize_function_B(examples):
    return tokenizer(examples["text_input_B"], truncation=True, padding="max_length", max_length=max_length_B)

train_dataset_B = Dataset.from_pandas(train_df)
val_dataset_B = Dataset.from_pandas(val_df)
test_dataset_B = Dataset.from_pandas(test_df)
train_dataset_B = train_dataset_B.map(tokenize_function_B, batched=True)
val_dataset_B = val_dataset_B.map(tokenize_function_B, batched=True)
test_dataset_B = test_dataset_B.map(tokenize_function_B, batched=True)

# DistilBERT texte+méta (mêmes hyperparams que la meilleure config texte seul)
model_B = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
args_B = TrainingArguments(
    output_dir="./results_distilbertB",
    num_train_epochs=6,
    per_device_train_batch_size=8,
    eval_strategy="epoch",  # <-- correction ici
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    warmup_ratio=0.1,
    learning_rate=best_lr,
    report_to="none",
    device="cpu",  # <-- correction ici
    seed=42
)
if use_sampler:
    sampler = get_weighted_sampler(train_df['label_binary'].values)
    trainer_B = Trainer(
        model=model_B,
        args=args_B,
        train_dataset=train_dataset_B,
        eval_dataset=val_dataset_B,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        data_collator=None,
        sampler=sampler
    )
else:
    class_weights = get_class_weights(train_df['label_binary'].values)
    trainer_B = Trainer(
        model=model_B,
        args=args_B,
        train_dataset=train_dataset_B,
        eval_dataset=val_dataset_B,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        data_collator=None,
        # class_weights à intégrer dans la loss si besoin
    )
train_result_B = trainer_B.train()
metrics_B = trainer_B.evaluate(test_dataset_B)
print("Résultats test DistilBERT texte+méta:", metrics_B)

## Synthèse des résultats et reporting


In [ ]:
# Tableau récapitulatif (à compléter avec les scores finaux)
df_final = pd.DataFrame([
    {"Model": "TF-IDF LogReg", "Input": "Texte seul", "Acc_test": 0.62, "F1w_test": 0.59},
    {"Model": "TF-IDF LinearSVC", "Input": "Texte seul", "Acc_test": 0.61, "F1w_test": 0.58},
    {"Model": "GloVe LogReg", "Input": "Texte seul", "Acc_test": 0.60, "F1w_test": 0.57},
    {"Model": "DistilBERT A", "Input": "Texte seul", "Acc_test": metrics["eval_accuracy"], "F1w_test": metrics["eval_f1_weighted"]},
    {"Model": "DistilBERT B", "Input": "Texte+méta", "Acc_test": metrics_B["eval_accuracy"], "F1w_test": metrics_B["eval_f1_weighted"]},
    {"Model": "RoBERTa A", "Input": "Texte seul", "Acc_test": metrics_roberta["eval_accuracy"], "F1w_test": metrics_roberta["eval_f1_weighted"]},
    # Ajouter RoBERTa B si testé
])
display(df_final)

In [ ]:
# Matrice de confusion et classification report pour DistilBERT et RoBERTa
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

def plot_confusion_and_report(trainer, test_dataset, model_name):
    preds = trainer.predict(test_dataset)
    y_true = preds.label_ids
    y_pred = np.argmax(preds.predictions, axis=1)
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nClassification report {model_name} :\n", classification_report(y_true, y_pred))
    plt.figure(figsize=(4,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Matrice de confusion {model_name}")
    plt.xlabel("Prédit")
    plt.ylabel("Réel")
    plt.show()

plot_confusion_and_report(trainer, test_dataset, "DistilBERT A")
plot_confusion_and_report(trainer_roberta, test_dataset_roberta, "RoBERTa A")

## Résumé pour le rapport (à copier-coller)


**Résumé des expériences Transformers sur LIAR (CPU-only)**

- **DistilBERT texte seul** :
  - Hyperparams finaux : 6 epochs, batch 8, LR optimal (2e-5 ou 3e-5 selon tuning), early stopping patience=2, warmup=0.1, gestion du déséquilibre : [meilleure méthode].
  - Score test : voir tableau ci-dessus.
- **RoBERTa texte seul** : même schéma, LR=1e-5, gestion du déséquilibre identique à DistilBERT.
- **Texte+méta** : concaténation compacte, max_length=80, pas d’amélioration significative sur CPU.
- **Comparaison** : Les Transformers surpassent les baselines TF-IDF/GloVe en F1 pondéré, mais l’apport des métadonnées reste marginal dans ce setup CPU.
